# Week 1 – Python for ML & Data Preprocessing

Same workflow as `preprocess.py`, split into cells so each step's output can be seen. Dataset: `data/employee_data_raw.csv` (synthetic, made for this task).

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

os.makedirs("figures", exist_ok=True)
os.makedirs("data", exist_ok=True)

TARGET = "Attrition"
NUM_COLS = ["Age", "Experience_Years", "Monthly_Income",
            "Satisfaction_Score", "Weekly_Work_Hours"]
CAT_COLS = ["Education", "Department", "City", "Job_Level"]


# small helper (basic Python: a function, a loop, a dict, an f-string)
def describe_missing(frame):
    """Print how many values are missing in each column that has any."""
    counts = frame.isnull().sum()
    for col, n in counts.items():
        if n > 0:
            pct = n / len(frame) * 100
            print(f"  {col:<20} {n:>3} missing ({pct:.1f}%)")

## 1. Load and inspect

Load the CSV and look at it before touching anything: shape, first rows, data types, missing values.

In [2]:
df = pd.read_csv("data/employee_data_raw.csv")
print("shape:", df.shape)
print(df.head().to_string())
print("\ndata types:")
print(df.dtypes.to_string())
print("\nmissing values:")
describe_missing(df)

shape: (508, 10)
    Age  Experience_Years  Monthly_Income Education Department       City Job_Level  Satisfaction_Score  Weekly_Work_Hours  Attrition
0  38.0              16.0         85184.0    Master         IT      Delhi    Junior                 4.2               42.5          0
1  26.0               2.0         26808.0   Diploma  Marketing    Kolkata       Mid                 6.5               34.9          0
2  25.0               2.0         29918.0   Diploma  Marketing  Hyderabad    Junior                 7.0               35.4          1
3  34.0              11.0         63178.0    Master    Finance  Hyderabad    Junior                 5.2               35.1          0
4  38.0              14.0         85081.0    Master    Finance     Mumbai       Mid                 6.6               44.5          0

data types:
Age                   float64
Experience_Years      float64
Monthly_Income        float64
Education                 str
Department                str
City            

## 2. Duplicates

Exact duplicate rows can double-count the same person, so they are dropped.

In [3]:
n_dup = df.duplicated().sum()
print("duplicate rows found:", n_dup)
df = df.drop_duplicates().reset_index(drop=True)
print("shape after dropping them:", df.shape)

duplicate rows found: 8
shape after dropping them: (500, 10)


## 3. Exploratory data analysis

Quick EDA: summary statistics, class balance, attrition by department, an outlier check with the IQR rule, and five plots saved in `figures/`.

In [4]:
print(df[NUM_COLS].describe().round(2).to_string())

print("\nattrition split (0 = stayed, 1 = left):")
print(df[TARGET].value_counts().to_string())
print(df[TARGET].value_counts(normalize=True).round(3).to_string())

print("\nattrition rate by department:")
print(df.groupby("Department")[TARGET].mean().round(3).to_string())

# outlier check with the IQR rule (just looking, not removing anything)
print("\npossible outliers (IQR rule):")
for col in NUM_COLS:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((df[col] < low) | (df[col] > high)).sum()
    print(f"  {col:<20} {n_out} outside [{low:.1f}, {high:.1f}]")

# plot 1: histograms of numeric columns
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, col in zip(axes.ravel(), NUM_COLS):
    ax.hist(df[col].dropna(), bins=20, color="#4C72B0", edgecolor="white")
    ax.set_title(col)
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.savefig("figures/01_numeric_histograms.png", dpi=110)
plt.close()

# plot 2: missing values per column
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values()
plt.figure(figsize=(7, 3.5))
plt.barh(missing.index, missing.values, color="#DD8452")
plt.xlabel("number of missing values")
plt.title("Missing values per column (after removing duplicates)")
plt.tight_layout()
plt.savefig("figures/02_missing_values.png", dpi=110)
plt.close()

# plot 3: attrition rate by department
rate = df.groupby("Department")[TARGET].mean().sort_values()
plt.figure(figsize=(7, 3.5))
plt.barh(rate.index, rate.values, color="#55A868")
plt.xlabel("attrition rate")
plt.title("Attrition rate by department")
plt.tight_layout()
plt.savefig("figures/03_attrition_by_department.png", dpi=110)
plt.close()

# plot 4: correlation heatmap (numeric columns + target)
corr = df[NUM_COLS + [TARGET]].corr()
plt.figure(figsize=(6.5, 5.5))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.title("Correlation matrix")
plt.tight_layout()
plt.savefig("figures/04_correlation_heatmap.png", dpi=110)
plt.close()
print("\nsaved 4 figures to figures/")

          Age  Experience_Years  Monthly_Income  Satisfaction_Score  Weekly_Work_Hours
count  485.00            500.00          482.00              488.00             500.00
mean    30.05              8.39        67084.42                6.51              42.01
std      6.64              6.66        31189.68                1.66               5.01
min     20.00              0.00        15000.00                1.60              30.00
25%     25.00              2.00        42441.00                5.30              38.38
50%     30.00              8.00        64837.50                6.50              41.90
75%     35.00             13.00        87249.00                7.70              45.22
max     51.00             29.00       180000.00               10.00              56.20

attrition split (0 = stayed, 1 = left):
Attrition
0    335
1    165
Attrition
0    0.67
1    0.33

attrition rate by department:
Department
Finance       0.266
HR            0.329
IT            0.398
Marketing     0.

![01_numeric_histograms.png](figures/01_numeric_histograms.png)

![02_missing_values.png](figures/02_missing_values.png)

![03_attrition_by_department.png](figures/03_attrition_by_department.png)

![04_correlation_heatmap.png](figures/04_correlation_heatmap.png)

## 4. Train / test split

Split **before** filling or scaling anything, so the test rows can't influence the numbers used for cleaning.

In [5]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("train:", X_train.shape, " test:", X_test.shape)

train: (400, 9)  test: (100, 9)


## 5. Missing values

Numeric gaps are filled with the median of the training data, categorical gaps with the most common training value. The same values are reused on the test set.

In [6]:
X_train = X_train.copy()
X_test = X_test.copy()

medians = X_train[NUM_COLS].median()
modes = X_train[CAT_COLS].mode().iloc[0]
print("medians used:", medians.round(2).to_dict())
print("modes used:  ", modes.to_dict())

X_train[NUM_COLS] = X_train[NUM_COLS].fillna(medians)
X_test[NUM_COLS] = X_test[NUM_COLS].fillna(medians)
X_train[CAT_COLS] = X_train[CAT_COLS].fillna(modes)
X_test[CAT_COLS] = X_test[CAT_COLS].fillna(modes)

print("missing left in train:", int(X_train.isnull().sum().sum()),
      "| in test:", int(X_test.isnull().sum().sum()))

medians used: {'Age': 30.0, 'Experience_Years': 7.0, 'Monthly_Income': 64226.0, 'Satisfaction_Score': 6.4, 'Weekly_Work_Hours': 41.7}
modes used:   {'Education': 'Bachelor', 'Department': 'IT', 'City': 'Bengaluru', 'Job_Level': 'Mid'}
missing left in train: 0 | in test: 0


## 6. Categorical encoding (one-hot)

One-hot encoding turns each category into its own 0/1 column. The test columns are forced to match the training columns.

In [7]:
X_train = pd.get_dummies(X_train, columns=CAT_COLS, dtype=int)
X_test = pd.get_dummies(X_test, columns=CAT_COLS, dtype=int)
# make sure test has exactly the same columns as train
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
print("columns before:", len(NUM_COLS) + len(CAT_COLS),
      "-> after:", X_train.shape[1])
print(list(X_train.columns))

columns before: 9 -> after: 23
['Age', 'Experience_Years', 'Monthly_Income', 'Satisfaction_Score', 'Weekly_Work_Hours', 'Education_Bachelor', 'Education_Diploma', 'Education_Master', 'Education_PhD', 'Department_Finance', 'Department_HR', 'Department_IT', 'Department_Marketing', 'Department_Operations', 'City_Bengaluru', 'City_Delhi', 'City_Hyderabad', 'City_Kolkata', 'City_Mumbai', 'Job_Level_Junior', 'Job_Level_Lead', 'Job_Level_Mid', 'Job_Level_Senior']


## 7. Scaling

Min-max scaling is shown on one column (and checked by hand with NumPy). Standardization is what is actually applied to the numeric features.

In [8]:
# 7a. Min-max normalization, shown on one column so the effect is visible
mm = MinMaxScaler()
income_mm = mm.fit_transform(X_train[["Monthly_Income"]])
print("Monthly_Income before: min", X_train["Monthly_Income"].min(),
      "max", X_train["Monthly_Income"].max())
print("Monthly_Income after min-max: min", income_mm.min(),
      "max", income_mm.max())

# same thing done by hand with numpy, to check I understand the formula
inc = X_train["Monthly_Income"].to_numpy()
by_hand = (inc - inc.min()) / (inc.max() - inc.min())
print("hand-written formula matches sklearn:",
      np.allclose(by_hand, income_mm.ravel()))

# 7b. Standardization (used for the actual processed data)
scaler = StandardScaler()
X_train[NUM_COLS] = scaler.fit_transform(X_train[NUM_COLS])
X_test[NUM_COLS] = scaler.transform(X_test[NUM_COLS])   # transform only!
print("\nafter standardization (train):")
print(X_train[NUM_COLS].agg(["mean", "std"]).round(3).to_string())

Monthly_Income before: min 15000.0 max 180000.0
Monthly_Income after min-max: min 0.0 max 1.0000000000000002
hand-written formula matches sklearn: True

after standardization (train):
        Age  Experience_Years  Monthly_Income  Satisfaction_Score  Weekly_Work_Hours
mean  0.000             0.000           0.000              -0.000              0.000
std   1.001             1.001           1.001               1.001              1.001


## 8. Feature selection

`SelectKBest` with the ANOVA F-test scores every feature against the target; the top 10 are kept.

In [9]:
selector = SelectKBest(score_func=f_classif, k=10)
selector.fit(X_train, y_train)

scores = pd.Series(selector.scores_, index=X_train.columns).sort_values(ascending=False)
print("F-scores, highest first:")
print(scores.round(2).to_string())

selected = X_train.columns[selector.get_support()].tolist()
print("\nkept 10 features:", selected)

plt.figure(figsize=(7, 5))
scores.sort_values().plot(kind="barh", color="#8172B2")
plt.xlabel("F-score")
plt.title("Feature scores (SelectKBest)")
plt.tight_layout()
plt.savefig("figures/05_feature_scores.png", dpi=110)
plt.close()

F-scores, highest first:
Satisfaction_Score       9.80
City_Mumbai              7.18
Monthly_Income           4.98
Experience_Years         4.42
Department_Finance       3.77
Age                      3.00
City_Kolkata             1.90
Education_PhD            1.56
Weekly_Work_Hours        1.15
Job_Level_Lead           0.98
Department_IT            0.84
City_Delhi               0.72
Department_Operations    0.64
Job_Level_Mid            0.57
Education_Master         0.43
City_Bengaluru           0.19
City_Hyderabad           0.18
Education_Diploma        0.14
Education_Bachelor       0.04
Department_HR            0.03
Job_Level_Junior         0.01
Department_Marketing     0.01
Job_Level_Senior         0.00

kept 10 features: ['Age', 'Experience_Years', 'Monthly_Income', 'Satisfaction_Score', 'Weekly_Work_Hours', 'Education_PhD', 'Department_Finance', 'City_Kolkata', 'City_Mumbai', 'Job_Level_Lead']


![05_feature_scores.png](figures/05_feature_scores.png)

## 9. Save processed data

Save the processed train/test data (all encoded + scaled columns plus the target) and the list of selected features.

In [10]:
train_out = X_train.copy()
train_out[TARGET] = y_train.values
test_out = X_test.copy()
test_out[TARGET] = y_test.values
train_out.to_csv("data/processed_train.csv", index=False)
test_out.to_csv("data/processed_test.csv", index=False)
pd.Series(selected, name="selected_feature").to_csv(
    "data/selected_features.csv", index=False)
print("\nsaved data/processed_train.csv, processed_test.csv, selected_features.csv")


saved data/processed_train.csv, processed_test.csv, selected_features.csv


## 10. Sanity check with a small model

A small logistic regression, only to prove the processed data can be used by a model.

In [11]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train[selected], y_train)
acc = accuracy_score(y_test, model.predict(X_test[selected]))
baseline = max(y_test.mean(), 1 - y_test.mean())   # always guess the majority class
print(f"logistic regression accuracy: {acc:.3f}")
print(f"always-predict-'stayed' baseline: {baseline:.3f}")
print("The model runs on the processed data, which is all this check is for.")
print("The synthetic data has only weak patterns, so accuracy is modest.")

logistic regression accuracy: 0.640
always-predict-'stayed' baseline: 0.670
The model runs on the processed data, which is all this check is for.
The synthetic data has only weak patterns, so accuracy is modest.


## What I took from this

- Look at the data first (shape, dtypes, missing, duplicates) before deciding what to do.
- Split first, then fit anything (medians, encoders, scalers) on the training part only.
- Scaling matters for models that compare feature sizes; one-hot columns are already 0/1 so I left them alone.
- The model result is not the point here. Accuracy (0.64) is below the always-'stayed' baseline (0.67), which is expected because the synthetic data has only weak patterns.